In [1]:
import sys
import os
os.chdir("/home/felixpersson/MasterThesisProject")
print(os.getcwd())

/home/felixpersson/MasterThesisProject


In [8]:
import json
import matplotlib.pyplot as plt
import numpy as np

import utils.sodium_properties as s_props
import data.dataclass as d_class

from pathlib import Path

from models.heatpipe.heat_pipe_limitations_model import HeatPipeLimitations
from models.heatpipe.liquid_discretised_model import LiquidDiscretised
from coupled.heatpipe import Heatpipe

from utils.solver import Solver
from utils.iso_reactor_utils import generate_config, generate_config_seq
from utils.interpolator import OpenMCTallyGridSurrogate

from coupled.vapour_reactor import VapourReactor
from coupled.iso_reactor import Reactor

In [ ]:
MODEL_PATH = Path("./utils/rgi_surrogate.joblib")
interpolator_model = OpenMCTallyGridSurrogate()
interpolator_model = interpolator_model.load(MODEL_PATH)

In [9]:
with open("./data/reactor_data.json", "r") as f:
    data = json.load(f)
    data["HeatPipe"]["material"]["h_cond"] = 550

geom = d_class.HeatpipeGeometry(**data["HeatPipe"]["geometry"])
mesh = d_class.HeatpipeMesh(N_R=20, N_Z=50)
mat = d_class.HeatpipeMaterial(**data["HeatPipe"]["material"])
bc = d_class.HeatpipeBC(**data["HeatPipe"]["bc"])
wick = d_class.HeatpipeWick(**data["HeatPipe"]["wick"])
cfg = d_class.HeatpipeConfig(geom, mesh, mat, wick, bc)
cfg_HP = cfg.resolve_geometry()

HP = Heatpipe(cfg_HP)
HP_limits = HeatPipeLimitations(HP, cfg_HP)

T_span = np.linspace(700, 1400, 20)
Q_entrainment = HP_limits.calculate_analytical_entrainment_limit(T_span)
Q_sonic = HP_limits.calculate_analytical_sonic_limit(T_span)
Q_boil = HP_limits.calculate_analytical_boiling_limit(T_span)

Geometry warning: delta_wick changed from 0.00038933333333333344 to 0.0003650000000000003
Geometry warning: delta_gap changed from 0.0006813333333333335 to 0.0007300000000000006
Geometry warning: delta_wall changed from 0.00038933333333333344 to 0.0003650000000000003
Geometry warning: l_evap changed from 1.5 to 1.518
Geometry warning: l_adiabatic changed from 0.2 to 0.184
Geometry warning: l_cond changed from 0.6 to 0.598
[6.94830831e+06 3.03617924e+06 1.43669722e+06 7.28020621e+05
 3.91438615e+05 2.21613988e+05 1.31265720e+05 8.09028981e+04
 5.16443826e+04 3.40096405e+04 2.30255582e+04 1.59790837e+04
 1.13368956e+04 8.20427251e+03 6.04375873e+03 4.52385983e+03
 3.43511656e+03 2.64221637e+03 2.05594910e+03 1.61638608e+03]
{'invalid_property_points': 0, 'no_real_root_points': 0, 'non_finite_result_points': 0}


In [10]:
def make_liquid_input(T_HP, T_v):
    """
    LiquidDiscretised expects one flat array:

        [flattened solid temperature field, scalar vapour temperature]

    The coupled Heatpipe solver may return T_HP either as:
        - flat array of shape (N_Z * N_R,)
        - 2D array of shape (N_Z, N_R)

    Therefore, we explicitly flatten it before appending T_v.
    """

    T_HP_flat = np.asarray(T_HP, dtype=float).reshape(-1)

    T_v_arr = np.asarray(T_v, dtype=float)
    T_v_scalar = float(np.mean(T_v_arr))

    T_liquid_input = np.concatenate([
        T_HP_flat,
        np.array([T_v_scalar], dtype=float),
    ])

    return T_liquid_input, T_v_scalar

In [ ]:
from utils import sodium_properties


with open("./data/reactor_data.json", "r") as f:
    data = json.load(f)
    data["HeatPipe"]["material"]["h_cond"] = 550
    data["Reactor"]["power"]["thermal"] = 15e6 # The power I want you to plot is thermal power / #number of fuel pins * ratio of HP to FP = P_th / 2970 * (24/7)

Ns = [15, 15, 20]
cfg = generate_config(data, *Ns)

iso_reactor      = Reactor(cfg)
vap_reactor_800  = VapourReactor(cfg)
vap_reactor      = VapourReactor(cfg)
vap_reactor_full = VapourReactor(cfg)

vap_reactor.heatpipe.solid.k_temperature = 1100

vap_reactor_full.set_interpolator_model(interpolator_model)
vap_reactor_full.set_variable_k(True)

solver_vap = Solver([vap_reactor_800, vap_reactor, vap_reactor_full], iterate = True)
solver_vap.fsolve()

((T_solid, u_v, T_v), T_FP, (phi_n_g, k)) = solver_vap.solutions[2]


T_liquid_input, T_v_scalar = make_liquid_input(T_solid, T_v)

liquid = LiquidDiscretised(cfg_HP, T_liquid_input)

P_l = liquid.get_pressure_drop_profile()
P_v = sodium_properties.calculate_Na_pressure_v(T_v)

P_l = np.asarray(P_l, dtype=float)
P_v = np.asarray(P_v, dtype=float)

if len(P_l) != len(P_v):
    raise ValueError(
        f"Pressure-profile length mismatch: "
        f"len(P_l)={len(P_l)}, len(P_v)={len(P_v)}"
    )

# Shift both profiles to start at zero.
P_v = P_v - P_v[0]
P_l = P_l - P_l[0]

# Reverse liquid pressure profile so it can be compared against the
# vapour profile in the same axial direction.
P_l_rev = P_l[::-1]

# Shift vapour profile until it just touches the liquid profile.
shift_touch = np.min(P_v - P_l_rev)
P_v_touch = P_v - shift_touch

# total pressure drop and velocity in mach. 
Delta_P_tot = P_v_touch - P_l_rev
u_mach = HP._calculate_c_s(T_v) 


Mesh warning: N_Z changed from 20 to 23
Mesh warning: Fuel pin N_Z changed from 9 to 15
Mesh warning: Neutronics N_Z changed from 9 to 15
Solving Heat pipe initial values
fsolve message: The solution converged.
Running iteration 1 ...
fsolve message: The solution converged.
Running iteration 2 ...
fsolve message: The solution converged.
Running iteration 3 ...
fsolve message: The solution converged.
Geometry warning: delta_wick changed from 0.00038933333333333344 to 0.0003650000000000003
Geometry warning: delta_gap changed from 0.0006813333333333335 to 0.0007300000000000006
Geometry warning: delta_wall changed from 0.00038933333333333344 to 0.0003650000000000003
Geometry warning: l_evap changed from 1.5 to 1.518
Geometry warning: l_adiabatic changed from 0.2 to 0.184
Geometry warning: l_cond changed from 0.6 to 0.598
!
!
!
[6.94830831e+06 3.03617924e+06 1.43669722e+06 7.28020621e+05
 3.91438615e+05 2.21613988e+05 1.31265720e+05 8.09028981e+04
 5.16443826e+04 3.40096405e+04 2.30255582e+

In [ ]:
from __future__ import annotations

import copy
import json
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional

import numpy as np

from utils import sodium_properties as s_props
from utils.solver import Solver


N_FUEL_PINS = 2970
HP_TO_FP_RATIO = 24.0 / 7.0


def thermal_power_to_plot_Q(P_th: float) -> float:
    """
    Converts total reactor thermal power to the heat-load value used in
    the Q--T limit plot.

    User definition:
        Q = P_th / 2970 * (24 / 7)
    """
    return float(P_th) / N_FUEL_PINS * HP_TO_FP_RATIO


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def temperature_scalar(T_v: np.ndarray, T_v_scalar_from_liquid_input=None) -> float:
    """
    Characteristic temperature stored as T_i in the Q--T limit data.

    Current choice:
        mean vapour temperature.

    Alternatives you may prefer:
        return float(np.max(T_v))
        return float(T_v[-1])
        return float(T_v_scalar_from_liquid_input)
    """
    return float(np.mean(np.asarray(T_v, dtype=float)))


def align_like_velocity(u_v: np.ndarray, cell_values: np.ndarray) -> np.ndarray:
    """
    Aligns cell-centred quantities, such as c_s(T_v), with the vapour
    velocity grid.

    Handles common cases:
        len(u_v) == len(T_v)
        len(u_v) == len(T_v) - 1
        len(u_v) == len(T_v) + 1

    Falls back to interpolation on a normalized axial coordinate.
    """
    u_v = np.asarray(u_v, dtype=float).reshape(-1)
    cell_values = np.asarray(cell_values, dtype=float).reshape(-1)

    n_u = len(u_v)
    n_c = len(cell_values)

    if n_u == n_c:
        return cell_values

    if n_u == n_c - 1:
        return 0.5 * (cell_values[:-1] + cell_values[1:])

    if n_u == n_c + 1:
        x_c = np.linspace(0.0, 1.0, n_c)
        x_u = np.linspace(0.0, 1.0, n_u)
        return np.interp(x_u, x_c, cell_values)

    x_c = np.linspace(0.0, 1.0, n_c)
    x_u = np.linspace(0.0, 1.0, n_u)
    return np.interp(x_u, x_c, cell_values)


def build_heatpipe_config_from_data(
    data: dict,
    *,
    N_R: int = 20,
    N_Z: int = 15,
    h_cond: float = 550.0,
):
    """
    Builds the standalone heat-pipe config used by LiquidDiscretised,
    Heatpipe, and HeatPipeLimitations.

    Important:
        N_Z should match the axial heat-pipe resolution used by the
        coupled reactor solution. If LiquidDiscretised complains about
        shape mismatch, this N_Z is the first thing to check.
    """
    data = copy.deepcopy(data)
    data["HeatPipe"]["material"]["h_cond"] = float(h_cond)

    geom = d_class.HeatpipeGeometry(**data["HeatPipe"]["geometry"])
    mesh = d_class.HeatpipeMesh(N_R=N_R, N_Z=N_Z)
    mat = d_class.HeatpipeMaterial(**data["HeatPipe"]["material"])
    bc = d_class.HeatpipeBC(**data["HeatPipe"]["bc"])
    wick = d_class.HeatpipeWick(**data["HeatPipe"]["wick"])

    cfg = d_class.HeatpipeConfig(geom, mesh, mat, wick, bc)
    cfg_HP = cfg.resolve_geometry()

    HP = Heatpipe(cfg_HP)

    return cfg_HP, HP


# ---------------------------------------------------------------------
# Data containers
# ---------------------------------------------------------------------

@dataclass
class ReactorLimitPoint:
    P_th: float
    h_cond: float
    Q_plot: float
    T_plot: float

    max_mach: float
    capillary_pressure_required: float
    capillary_pressure_max: float
    capillary_margin: float

    T_v: np.ndarray
    T_solid: np.ndarray
    u_v: np.ndarray
    mach_profile: np.ndarray
    delta_p_profile: np.ndarray


@dataclass
class RootResult:
    point: Optional[ReactorLimitPoint]
    residual: float
    status: str
    n_solves: int


# ---------------------------------------------------------------------
# Main solver
# ---------------------------------------------------------------------

class ReactorHeatPipeLimitSolver:
    """
    Computes analytic and numerical heat-pipe operating limits for the
    coupled reactor model.

    Analytic limits:
        - sonic
        - entrainment
        - boiling

    Numerical limits:
        - sonic
        - capillary

    The numerical curves are generated by sweeping condenser h values.
    For each h_cond, the reactor thermal power is solved by bisection
    until the chosen limit criterion is reached.
    """

    def __init__(
        self,
        *,
        reactor_data_path: str | Path = "./data/reactor_data.json",
        Ns=(15, 15, 20),
        cfg_HP=None,
        HP=None,
        interpolator_model=None,
        h_cond_values=None,
        power_bounds=(1.0e6, 80.0e6),
        max_power=200.0e6,
        max_root_iter: int = 8,
        q_tol_rel: float = 0.02,
        analytic_T_span=(700.0, 1400.0),
        n_analytic_T: int = 100,
        output_path: str | Path = "./heatpipe_limit_curves.npz",
        checkpoint_path: str | Path = "./heatpipe_limit_curves_checkpoint.npz",
        store_profiles: bool = True,
    ):
        self.reactor_data_path = Path(reactor_data_path)
        self.Ns = tuple(Ns)

        with open(self.reactor_data_path, "r") as f:
            self.base_data = json.load(f)

        if cfg_HP is None or HP is None:
            cfg_HP, HP = build_heatpipe_config_from_data(
                self.base_data,
                N_R=20,
                N_Z=self.Ns[0],
                h_cond=550.0,
            )

        self.cfg_HP = cfg_HP
        self.HP = HP
        self.interpolator_model = interpolator_model

        if h_cond_values is None:
            # This is intentionally quite small. Each point requires a root solve,
            # and each reactor solve is assumed to take about 15 seconds.
            h_cond_values = np.linspace(300.0, 1200.0, 8)

        self.h_cond_values = np.asarray(h_cond_values, dtype=float)

        self.power_bounds = tuple(float(x) for x in power_bounds)
        self.max_power = float(max_power)
        self.max_root_iter = int(max_root_iter)
        self.q_tol_rel = float(q_tol_rel)

        self.analytic_T_span = tuple(float(x) for x in analytic_T_span)
        self.n_analytic_T = int(n_analytic_T)

        self.output_path = Path(output_path)
        self.checkpoint_path = Path(checkpoint_path)

        self.store_profiles = bool(store_profiles)

        self._cache: dict[tuple[float, float], ReactorLimitPoint] = {}

    # -----------------------------------------------------------------
    # Coupled reactor solve
    # -----------------------------------------------------------------

    def solve_reactor_point(self, P_th: float, h_cond: float) -> ReactorLimitPoint:
        """
        Runs the full coupled reactor at a given thermal power and condenser
        heat-transfer coefficient.
        """
        P_th = float(P_th)
        h_cond = float(h_cond)

        cache_key = (round(P_th, 6), round(h_cond, 6))
        if cache_key in self._cache:
            return self._cache[cache_key]

        data = copy.deepcopy(self.base_data)
        data["HeatPipe"]["material"]["h_cond"] = h_cond
        data["Reactor"]["power"]["thermal"] = P_th

        cfg = generate_config(data, *self.Ns)

        iso_reactor = Reactor(cfg)
        vap_reactor_800 = VapourReactor(cfg)
        vap_reactor = VapourReactor(cfg)
        vap_reactor_full = VapourReactor(cfg)

        vap_reactor.heatpipe.solid.k_temperature = 1100

        if self.interpolator_model is not None:
            vap_reactor_full.set_interpolator_model(self.interpolator_model)

        vap_reactor_full.set_variable_k(True)

        solver_vap = Solver(
            [
                vap_reactor_800,
                vap_reactor,
                vap_reactor_full,
            ],
            iterate=True,
        )

        solver_vap.fsolve()

        ((T_solid, u_v, T_v), T_FP, (phi_n_g, k)) = solver_vap.solutions[2]

        T_solid = np.asarray(T_solid, dtype=float)
        u_v = np.asarray(u_v, dtype=float)
        T_v = np.asarray(T_v, dtype=float)

        # -------------------------------------------------------------
        # Mach number
        # -------------------------------------------------------------
        c_s = np.asarray(self.HP._calculate_c_s(T_v), dtype=float)
        c_s_on_u_grid = align_like_velocity(u_v, c_s)

        mach_profile = np.abs(u_v.reshape(-1)) / c_s_on_u_grid.reshape(-1)
        max_mach = float(np.nanmax(mach_profile))

        # -------------------------------------------------------------
        # Liquid/vapour pressure profiles for capillary criterion
        # -------------------------------------------------------------
        T_liquid_input, T_v_scalar_from_liquid = make_liquid_input(T_solid, T_v)

        liquid = LiquidDiscretised(self.cfg_HP, T_liquid_input)

        P_l = liquid.get_pressure_drop_profile()
        P_v = s_props.calculate_Na_pressure_v(T_v)

        P_l = np.asarray(P_l, dtype=float).reshape(-1)
        P_v = np.asarray(P_v, dtype=float).reshape(-1)

        if len(P_l) != len(P_v):
            raise ValueError(
                "Pressure-profile length mismatch: "
                f"len(P_l)={len(P_l)}, len(P_v)={len(P_v)}. "
                "Check that cfg_HP has the same axial mesh as the reactor heat-pipe model."
            )

        # Shift both profiles to start at zero.
        P_v = P_v - P_v[0]
        P_l = P_l - P_l[0]

        # Reverse liquid pressure profile so it can be compared against the
        # vapour profile in the same axial direction.
        P_l_rev = P_l[::-1]

        # Shift vapour profile until it just touches the liquid profile.
        shift_touch = np.min(P_v - P_l_rev)
        P_v_touch = P_v - shift_touch

        # Required capillary pressure profile.
        delta_p_profile = P_v_touch - P_l_rev
        capillary_pressure_required = float(np.nanmax(delta_p_profile))

        T_cap = temperature_scalar(T_v, T_v_scalar_from_liquid)
        sigma_l = s_props.calculate_Na_surface_tension(T_cap)

        capillary_pressure_max = float(
            2.0 * sigma_l / self.cfg_HP.wick.r_pore
        )

        capillary_margin = capillary_pressure_max - capillary_pressure_required

        T_plot = temperature_scalar(T_v, T_v_scalar_from_liquid)
        Q_plot = thermal_power_to_plot_Q(P_th)

        point = ReactorLimitPoint(
            P_th=P_th,
            h_cond=h_cond,
            Q_plot=Q_plot,
            T_plot=T_plot,
            max_mach=max_mach,
            capillary_pressure_required=capillary_pressure_required,
            capillary_pressure_max=capillary_pressure_max,
            capillary_margin=capillary_margin,
            T_v=T_v,
            T_solid=T_solid,
            u_v=u_v,
            mach_profile=mach_profile,
            delta_p_profile=delta_p_profile,
        )

        self._cache[cache_key] = point
        return point

    # -----------------------------------------------------------------
    # Residual definitions
    # -----------------------------------------------------------------

    @staticmethod
    def sonic_residual(point: ReactorLimitPoint) -> float:
        """
        Sonic limit residual.

        Below limit:
            residual < 0

        At limit:
            residual = 0

        Above limit:
            residual > 0
        """
        return point.max_mach - 1.0

    @staticmethod
    def capillary_residual(point: ReactorLimitPoint) -> float:
        """
        Capillary limit residual.

        Below limit:
            required pressure < max capillary pressure
            residual < 0

        At limit:
            residual = 0

        Above limit:
            residual > 0
        """
        return (
            point.capillary_pressure_required
            - point.capillary_pressure_max
        )

    # -----------------------------------------------------------------
    # One-dimensional power root solve
    # -----------------------------------------------------------------

    def solve_power_for_limit(
        self,
        *,
        h_cond: float,
        residual_fn: Callable[[ReactorLimitPoint], float],
        limit_name: str,
    ) -> RootResult:
        """
        For a fixed h_cond, solve for the thermal power where residual_fn = 0.
        """
        P_low, P_high = self.power_bounds
        n_solves = 0

        def eval_residual(P):
            nonlocal n_solves
            point = self.solve_reactor_point(P, h_cond)
            n_solves += 1
            return residual_fn(point), point

        # Initial bracket.
        try:
            f_low, point_low = eval_residual(P_low)
            f_high, point_high = eval_residual(P_high)
        except Exception as exc:
            return RootResult(
                point=None,
                residual=np.nan,
                status=f"{limit_name}: failed during initial bracket: {exc}",
                n_solves=n_solves,
            )

        # If even the low power is above the limit, the root is below P_low.
        if f_low >= 0.0:
            return RootResult(
                point=point_low,
                residual=f_low,
                status=f"{limit_name}: already above limit at lower power bound",
                n_solves=n_solves,
            )

        # Expand upper bracket if needed, but do not exceed max_power.
        while f_high < 0.0 and P_high < self.max_power:
            P_high = min(1.5 * P_high, self.max_power)

            try:
                f_high, point_high = eval_residual(P_high)
            except Exception as exc:
                return RootResult(
                    point=point_high,
                    residual=f_high,
                    status=f"{limit_name}: failed while expanding upper bracket: {exc}",
                    n_solves=n_solves,
                )

        if f_high < 0.0:
            return RootResult(
                point=point_high,
                residual=f_high,
                status=f"{limit_name}: no crossing found below max_power",
                n_solves=n_solves,
            )

        # Bisection.
        best_point = point_high
        best_residual = f_high

        for _ in range(self.max_root_iter):
            P_mid = 0.5 * (P_low + P_high)

            try:
                f_mid, point_mid = eval_residual(P_mid)
            except Exception as exc:
                return RootResult(
                    point=best_point,
                    residual=best_residual,
                    status=f"{limit_name}: failed during bisection: {exc}",
                    n_solves=n_solves,
                )

            if abs(f_mid) < abs(best_residual):
                best_point = point_mid
                best_residual = f_mid

            if f_mid >= 0.0:
                P_high = P_mid
                f_high = f_mid
                point_high = point_mid
            else:
                P_low = P_mid
                f_low = f_mid
                point_low = point_mid

            P_scale = max(abs(P_mid), 1.0)
            if abs(P_high - P_low) / P_scale < self.q_tol_rel:
                break

        return RootResult(
            point=best_point,
            residual=best_residual,
            status=f"{limit_name}: converged",
            n_solves=n_solves,
        )

    # -----------------------------------------------------------------
    # Analytic limits
    # -----------------------------------------------------------------

    def compute_analytic_limits(self) -> dict[str, np.ndarray]:
        """
        Computes analytic sonic, entrainment, and boiling limits.
        """
        T_span = np.linspace(
            self.analytic_T_span[0],
            self.analytic_T_span[1],
            self.n_analytic_T,
        )

        HP_limits = HeatPipeLimitations(self.HP, self.cfg_HP)

        Q_sonic = HP_limits.calculate_analytical_sonic_limit(T_span)
        Q_entrainment = HP_limits.calculate_analytical_entrainment_limit(T_span)
        Q_boiling = HP_limits.calculate_analytical_boiling_limit(T_span)

        out = {
            "analytic_T": np.asarray(T_span, dtype=float),
            "sonic_analytic_Q": np.asarray(Q_sonic, dtype=float),
            "entrainment_analytic_Q": np.asarray(Q_entrainment, dtype=float),
            "boiling_analytic_Q": np.asarray(Q_boiling, dtype=float),
        }

        # Optional, only if you later add this method to HeatPipeLimitations.
        if hasattr(HP_limits, "calculate_analytical_capillary_limit"):
            out["capillary_analytic_Q"] = np.asarray(
                HP_limits.calculate_analytical_capillary_limit(T_span),
                dtype=float,
            )

        return out

    # -----------------------------------------------------------------
    # Numeric curves
    # -----------------------------------------------------------------

    def compute_numeric_curve(
        self,
        *,
        limit_name: str,
        residual_fn: Callable[[ReactorLimitPoint], float],
    ) -> dict[str, np.ndarray]:
        """
        Computes a numerical limit curve by sweeping h_cond and solving
        thermal power for each h_cond.
        """
        points: list[Optional[ReactorLimitPoint]] = []
        residuals = []
        statuses = []
        n_solves = []

        t0 = time.time()

        for j, h_cond in enumerate(self.h_cond_values):
            print(
                f"\n[{limit_name}] h_cond point {j + 1}/{len(self.h_cond_values)}: "
                f"h_cond={h_cond:.6g}"
            )

            result = self.solve_power_for_limit(
                h_cond=float(h_cond),
                residual_fn=residual_fn,
                limit_name=limit_name,
            )

            points.append(result.point)
            residuals.append(result.residual)
            statuses.append(result.status)
            n_solves.append(result.n_solves)

            if result.point is None:
                print(f"  status: {result.status}")
            else:
                print(
                    f"  status: {result.status}\n"
                    f"  P_th  = {result.point.P_th:.6e} W\n"
                    f"  Q     = {result.point.Q_plot:.6e} W\n"
                    f"  T     = {result.point.T_plot:.6f} K\n"
                    f"  resid = {result.residual:.6e}\n"
                    f"  solves this point = {result.n_solves}"
                )

            elapsed = time.time() - t0
            print(f"  elapsed for {limit_name}: {elapsed / 60.0:.2f} min")

        prefix = f"{limit_name}_numeric"

        data = {
            f"{prefix}_h_cond": np.asarray(self.h_cond_values, dtype=float),
            f"{prefix}_P_th": np.asarray(
                [p.P_th if p is not None else np.nan for p in points],
                dtype=float,
            ),
            f"{prefix}_Q": np.asarray(
                [p.Q_plot if p is not None else np.nan for p in points],
                dtype=float,
            ),
            f"{prefix}_T": np.asarray(
                [p.T_plot if p is not None else np.nan for p in points],
                dtype=float,
            ),
            f"{prefix}_residual": np.asarray(residuals, dtype=float),
            f"{prefix}_status": np.asarray(statuses, dtype=str),
            f"{prefix}_n_solves": np.asarray(n_solves, dtype=int),
        }

        if limit_name == "sonic":
            data[f"{prefix}_max_mach"] = np.asarray(
                [p.max_mach if p is not None else np.nan for p in points],
                dtype=float,
            )

        if limit_name == "capillary":
            data[f"{prefix}_pressure_required"] = np.asarray(
                [
                    p.capillary_pressure_required if p is not None else np.nan
                    for p in points
                ],
                dtype=float,
            )
            data[f"{prefix}_pressure_max"] = np.asarray(
                [
                    p.capillary_pressure_max if p is not None else np.nan
                    for p in points
                ],
                dtype=float,
            )
            data[f"{prefix}_margin"] = np.asarray(
                [
                    p.capillary_margin if p is not None else np.nan
                    for p in points
                ],
                dtype=float,
            )

        if self.store_profiles:
            data[f"{prefix}_T_v_profiles"] = np.asarray(
                [p.T_v if p is not None else np.array([]) for p in points],
                dtype=object,
            )
            data[f"{prefix}_mach_profiles"] = np.asarray(
                [p.mach_profile if p is not None else np.array([]) for p in points],
                dtype=object,
            )
            data[f"{prefix}_delta_p_profiles"] = np.asarray(
                [p.delta_p_profile if p is not None else np.array([]) for p in points],
                dtype=object,
            )

        return data

    # -----------------------------------------------------------------
    # Save
    # -----------------------------------------------------------------

    def save_results(self, results: dict[str, np.ndarray], path: str | Path):
        metadata = {
            "reactor_data_path": str(self.reactor_data_path),
            "Ns": self.Ns,
            "h_cond_values": self.h_cond_values.tolist(),
            "power_bounds": self.power_bounds,
            "max_power": self.max_power,
            "max_root_iter": self.max_root_iter,
            "q_tol_rel": self.q_tol_rel,
            "Q_conversion": "Q = P_th / 2970 * (24 / 7)",
            "numeric_sonic_criterion": "max(abs(u_v) / c_s(T_v)) = 1",
            "numeric_capillary_criterion": "max(delta_p_profile) = 2*sigma/r_pore",
            "T_plot_definition": "mean vapour temperature by default",
            "store_profiles": self.store_profiles,
        }

        results = dict(results)
        results["metadata_json"] = np.asarray(json.dumps(metadata, indent=2))

        np.savez(path, **results)

    # -----------------------------------------------------------------
    # Full run
    # -----------------------------------------------------------------

    def run(self) -> dict[str, np.ndarray]:
        """
        Runs all requested limits and stores them in output_path.
        """
        results: dict[str, np.ndarray] = {}

        print("\nComputing analytic heat-pipe limits...")
        analytic = self.compute_analytic_limits()
        results.update(analytic)
        self.save_results(results, self.checkpoint_path)

        print("\nComputing numerical sonic limit...")
        sonic_numeric = self.compute_numeric_curve(
            limit_name="sonic",
            residual_fn=self.sonic_residual,
        )
        results.update(sonic_numeric)
        self.save_results(results, self.checkpoint_path)

        print("\nComputing numerical capillary limit...")
        capillary_numeric = self.compute_numeric_curve(
            limit_name="capillary",
            residual_fn=self.capillary_residual,
        )
        results.update(capillary_numeric)

        self.save_results(results, self.output_path)

        print(f"\nSaved final limit data to: {self.output_path}")
        print(f"Saved checkpoint data to:  {self.checkpoint_path}")

        return results


In [ ]:
# Recommended starting point:
#
# 8 h-values * about 10 reactor solves * 2 numerical limits
# = roughly 160 full reactor solves.
#
# At 15 s per solve, this is about 40 min, plus overhead.
#
# Increase h_cond_values or max_root_iter only after confirming that
# the solver behaves sensibly.

with open("./data/reactor_data.json", "r") as f:
    base_data = json.load(f)

# Build cfg_HP and HP for the limit/pressure calculations.
# Make sure N_Z matches the heat-pipe axial mesh used by the reactor solve.
cfg_HP, HP = build_heatpipe_config_from_data(
    base_data,
    N_R=20,
    N_Z=15,
    h_cond=550.0,
)

limit_solver = ReactorHeatPipeLimitSolver(
    reactor_data_path="./data/reactor_data.json",
    Ns=(15, 15, 20),
    cfg_HP=cfg_HP,
    HP=HP,
    interpolator_model=interpolator_model,

    # These h-values move the reactor along different operating temperatures.
    h_cond_values=np.linspace(300.0, 1200.0, 8),

    # Adjust if the limit is outside this interval.
    power_bounds=(1.0e6, 80.0e6),
    max_power=200.0e6,

    # 8 bisection iterations is usually enough for a limit curve.
    max_root_iter=8,
    q_tol_rel=0.02,

    analytic_T_span=(700.0, 1400.0),
    n_analytic_T=100,

    output_path="./heatpipe_limit_curves.npz",
    checkpoint_path="./heatpipe_limit_curves_checkpoint.npz",
    store_profiles=True,
)

results = limit_solver.run()